In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, f1_score, accuracy_score, recall_score, precision_score, cohen_kappa_score
import seaborn as sns
from pathlib import Path
import os

In [ ]:
sds_dir = Path('/Users/lukashat/sds_mount/sds/sd22c003/phenotyping_benchmark/visual_qc/corrected_annotations/')
dataset = 'cHL_2_MIBI'

In [ ]:
# Each annotator has one file per crop: *_reannotated.csv (Exp1, Lukas) and *_reannotated_Philippe.csv (Exp2).
# The final expert label is the NewAnnotation column; NaN means the expert kept the original label.
# We deliberately do NOT use the Reannotated flag: an expert may have flagged a cell and then switched back to the original.

def load_annotations(folder, suffix):
    dfs = []
    for csv in sorted(folder.glob(f'*{suffix}')):
        df = pd.read_csv(csv)
        df['crop'] = csv.name.removesuffix(suffix)
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True) if dfs else None

def expert_labels(df, annotator):
    new_cols = [c for c in df.columns if 'new' in c.lower() and 'annot' in c.lower() and annotator.lower() in c.lower()]
    assert len(new_cols) == 1, f'Expected one new-annotation column for {annotator}, found {new_cols}'
    labels = df[new_cols[0]].fillna(df['cell_type'])
    n_switched_back = (df['Reannotated'] & (labels == df['cell_type'])).sum()
    print(f"{annotator}: using '{new_cols[0]}'; {n_switched_back} cells flagged Reannotated but final label == original")
    return labels

exp1 = load_annotations(sds_dir / dataset, '_reannotated.csv')
exp2 = load_annotations(sds_dir / dataset, '_reannotated_Philippe.csv')

key = ['crop', 'cell_id']
full_df = exp1.assign(Exp1=expert_labels(exp1, 'Lukas'))
if exp2 is not None:
    exp2 = exp2.assign(Exp2=expert_labels(exp2, 'Philippe'))
    full_df = full_df.merge(exp2[key + ['cell_type', 'Exp2']], on=key, how='left',
                            suffixes=('', '_exp2'), validate='one_to_one')
    assert full_df['Exp2'].notna().all(), 'Some cells are missing in the Philippe annotations'
    assert (full_df['cell_type'] == full_df['cell_type_exp2']).all(), 'Original cell_type differs between annotator files'
    full_df = full_df.drop(columns='cell_type_exp2')
else:
    full_df['Exp2'] = np.nan

In [ ]:
full_df

In [ ]:
# Expert review agreement: GT (cell_type) vs Exp1, GT vs Exp2, and Exp1 vs Exp2 (inter-expert)

def agreement_report(df, ref_col, cmp_col):
    agree = df[ref_col] == df[cmp_col]
    kappa = cohen_kappa_score(df[ref_col], df[cmp_col])
    print(f"\n===== {cmp_col} vs {ref_col}: {agree.mean():.1%} agreement "
          f"({agree.sum()} / {len(df)} cells), Cohen's kappa = {kappa:.3f}\n")

    per_type = (
        agree.groupby(df[ref_col])
        .agg(n_cells='count', n_agree='sum')
        .assign(n_changed=lambda d: d.n_cells - d.n_agree,
                pct_changed=lambda d: d.n_changed / d.n_cells * 100)
        .drop(columns='n_agree')
        .sort_values('pct_changed', ascending=False)
    )
    print(f"Disagreements per {ref_col} label:")
    print(per_type.to_string())

    changed = df[~agree]
    if not changed.empty:
        print(f"\n{ref_col} → {cmp_col} (n={len(changed)}):")
        print(changed.groupby([ref_col, cmp_col]).size().rename('count').reset_index().to_string(index=False))
    return agree.mean()

has_exp2 = full_df['Exp2'].notna().all()
exp1_acc  = agreement_report(full_df, 'cell_type', 'Exp1')
exp2_acc  = agreement_report(full_df, 'cell_type', 'Exp2') if has_exp2 else np.nan
inter_acc = agreement_report(full_df, 'Exp1', 'Exp2') if has_exp2 else np.nan

In [ ]:
# Collected across runs: change `dataset` above and re-run to add the next dataset
if 'agreement' not in globals():
    agreement = {}
agreement[dataset] = {
    'Expert_1_agreement': exp1_acc,
    'Expert_2_agreement': exp2_acc,
    'Inter_expert_agreement': inter_acc,
}

In [ ]:
agreement

In [ ]:
datasets = ['cHL_2_MIBI', 'IMMUcan', 'cHL_1_MIBI', 'cHL_CODEX', 'myeloma_IMC', 'Mouse Heart', 'healthy_bone_marrow', 'feto_maternal',
            'cycif_ovca', 'Breast_Jun0524', 'CRC_Feb2623', 'GVHD_Feb1323', 'Melanoma_Oct2522']
metrics = ['Expert_1_agreement', 'Expert_2_agreement', 'Inter_expert_agreement']
df = (
    pd.DataFrame.from_dict(agreement, orient='index')
    .reindex(index=datasets, columns=metrics)
    .rename_axis('Dataset')
    .reset_index()
)

In [ ]:
df

In [ ]:
fig, ax = plt.subplots(figsize=(2, 4))
sns.set_style('ticks')

sns.barplot(
    data=df, x='Dataset', y='Expert Agreement',
    edgecolor='black', linewidth=1.2,
    width=0.6, color='steelblue', ax=ax
)
ax.set_xlabel('')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=10)
ax.set_ylabel('Expert Agreement', fontsize=12)
sns.despine()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 1.4))
cmap = plt.cm.RdYlGn.copy()
cmap.set_bad('grey')
sns.heatmap(
    df.set_index('Dataset')[metrics].T,
    annot=True, fmt=".1%", cmap=cmap, cbar=False, linewidths=0.5, linecolor='black', vmax=1.0, vmin=0.0, annot_kws={"size": 8, "rotation": 90},
    ax=ax
)
ax.set_ylim(len(metrics) + 0.02, -0.02)
ax.set_xlim(-0.02, len(df) + 0.02)
ax.set_xlabel('')
ax.set_yticklabels(ax.get_yticklabels(), rotation=45, ha='right', rotation_mode='anchor')
fig.tight_layout()
fig.savefig('/Users/lukashat/Documents/PhD_Schapiro/Projects/phenotype_benchmark/stage_1/revision/plots/expert_agreement_heatmap.svg',bbox_inches='tight', dpi=300)

# CLaude clustering

In [35]:
df_pred = pd.read_csv('/Users/lukashat/Documents/PhD_Schapiro/Projects/phenotype_benchmark/stage_1/revision/claude_cluster/IMMUcan_predicted_phenotypes_res2_t3.csv')
df_true = pd.read_csv('/Users/lukashat/Documents/PhD_Schapiro/Projects/phenotype_benchmark/stage_1/revision/claude_cluster/IMMUcan_quantification.csv')

In [28]:
list(df_true.cell_type.unique())

['M2_Macrophage',
 'Plasma_cell',
 'Stroma',
 'Cancer',
 'Dendritic_cell',
 'undefined',
 'CD4+_T_cell',
 'NK_cell',
 'CD8+_T_cell',
 'Treg',
 'Neutrophil',
 'B_cell',
 'BnT',
 'Plasmacytoid_dendritic_cell']

In [23]:
df_pred

,cell_id,sample_id,x,y,leiden_res2,predicted_phenotype
0,1,1,326,2,0,undefined
1,2,1,420,2,10,CD4+_T_cell
2,4,1,1386,2,18,CD4+_T_cell
3,5,1,1436,2,32,Endothelial
4,6,1,1452,2,29,undefined
...,...,...,...,...,...,...
230884,40526,6,2301,2559,18,CD4+_T_cell
230885,40527,6,2351,2559,10,CD4+_T_cell
230886,40528,6,2371,2559,3,M2_Macrophage
230887,40529,6,2469,2559,3,M2_Macrophage


In [24]:
df_true

,CD45,CD20,dsDNA,pSLP-76,SLP-76,anti-H2AX (pS139),CD163,Histone H3,CD45RO,CD28,...,CD25,Na-K ATPase,cell_id,cellSize,x,y,sample_id,level_1_cell_type,level_2_cell_type,cell_type
0,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.077281,0.000000,0.000000,...,0.000000,0.000000,1,13,326,2,1,unedfined,undefined,undefined
1,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.167179,0.000000,0.000000,...,0.000000,0.000000,2,23,420,2,1,Immune,Lymphoid_immune,CD4+_T_cell
2,0.002631,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.176009,0.000106,0.000000,...,0.000769,0.003703,4,21,1386,2,1,Immune,Lymphoid_immune,CD4+_T_cell
3,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.077354,0.000000,0.000000,...,0.000000,0.000000,5,14,1436,2,1,Stromal,Vascular,Endothelial
4,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.182809,0.000000,0.000000,...,0.000000,0.000000,6,24,1452,2,1,unedfined,undefined,undefined
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
230884,0.001647,0.0,0.000260,0.000000,0.000000,0.0,0.000000,0.136305,0.009310,0.000000,...,0.004148,0.005979,40526,25,2301,2559,6,Immune,Lymphoid_immune,CD4+_T_cell
230885,0.000000,0.0,0.000694,0.000000,0.000000,0.0,0.009834,0.158602,0.002389,0.000000,...,0.000000,0.004477,40527,40,2351,2559,6,Immune,Lymphoid_immune,CD4+_T_cell
230886,0.000000,0.0,0.000453,0.002904,0.000779,0.0,0.003010,0.209390,0.007215,0.000000,...,0.000000,0.004282,40528,22,2371,2559,6,Immune,Lymphoid_immune,CD4+_T_cell
230887,0.000000,0.0,0.000000,0.000000,0.000524,0.0,0.040630,0.227213,0.003365,0.000000,...,0.000000,0.000000,40529,16,2469,2559,6,Immune,Lymphoid_immune,CD4+_T_cell


In [36]:
df_pred['true_phenotype'] = df_true['cell_type']

In [37]:
print(classification_report(df_pred['true_phenotype'], df_pred['predicted_phenotype'], zero_division=0
))

                             precision    recall  f1-score   support

                     B_cell       0.51      0.79      0.62      8263
                        BnT       0.52      0.44      0.48     13497
                CD4+_T_cell       0.62      0.58      0.60     25832
                CD8+_T_cell       0.80      0.65      0.72     35867
                     Cancer       0.95      0.75      0.84    210853
             Dendritic_cell       0.21      0.25      0.23     10098
              M2_Macrophage       0.56      0.64      0.60     33882
                    NK_cell       0.43      0.45      0.44      3412
                 Neutrophil       0.60      0.96      0.74     14723
                Plasma_cell       0.78      0.77      0.77     17972
Plasmacytoid_dendritic_cell       0.37      0.62      0.46      2915
                     Stroma       0.60      0.79      0.68     50127
                       Treg       0.83      0.74      0.78     10809
                  undefined      